# Sentiment Classification Project

In [15]:
import ctypes
ctypes.CDLL("/cluster/data/cuda/12.8.1/lib64/libcudart.so", mode=ctypes.RTLD_GLOBAL)
ctypes.CDLL("/cluster/data/cuda/12.8.1/lib64/libnvrtc.so", mode=ctypes.RTLD_GLOBAL)
ctypes.CDLL("/cluster/data/cuda/12.8.1/lib64/libnvJitLink.so", mode=ctypes.RTLD_GLOBAL)

import sys
sys.path.insert(0, "/home/taekim/.local/lib/python3.14/site-packages")

import torch
print(torch.__file__)
print(torch.version.cuda)
x = torch.randn(3, 3).cuda()
print(x @ x)

/home/taekim/.local/lib/python3.14/site-packages/torch/__init__.py
12.8
tensor([[ 1.1009,  0.5309,  0.4900],
        [-0.9737, -0.5863,  1.7528],
        [-0.0999, -1.0693, -0.3719]], device='cuda:0')


In [16]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [17]:
import torch
x = torch.randn(10, 10).cuda()
print(x @ x)

tensor([[ 0.0334, -2.4346, -2.4588,  0.4777,  1.3989, -3.2002, -4.8951, -2.6435,
          0.8559, -1.1835],
        [ 0.2579,  4.4442,  0.4662, -1.6443, -1.5665, -0.7767,  2.6105,  2.6358,
          0.3349,  0.8170],
        [ 0.4610, -2.4106, -1.4794,  2.2191,  0.3506,  2.9953, -2.1907,  0.7724,
         -1.9425,  0.4763],
        [ 3.3514,  1.3133, -2.2684,  1.2492,  1.8397,  4.3489, -0.6926, -0.6595,
         -0.2186, -1.4260],
        [ 1.5920, -0.6657, -4.1821, -1.4933, -1.1351,  1.8362,  1.7620, -1.0976,
          0.2813, -2.9781],
        [-2.4142, -2.7258,  8.8563,  0.4133, -1.0117, -1.2636, -6.0426,  4.9830,
         -6.1012,  0.6509],
        [ 1.2099, -2.6509,  0.4041,  2.3606,  1.8803,  2.2301, -7.1761, -0.0991,
         -1.0368,  0.1344],
        [-0.6273, -2.9002,  1.8344,  1.7998,  0.8740, -0.1506, -5.3777, -1.1832,
          2.2923,  1.0799],
        [ 0.5127, -0.1140, -1.0003, -0.6105,  1.7507,  0.4189,  2.0717, -2.5302,
          1.9591,  1.8157],
        [-0.2690,  

# Verify Setup
Make sure cuda (GPU) is available

In [18]:
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

CUDA Available: True
Device Name: NVIDIA GeForce RTX 5060 Ti


# Load data

In [19]:
train_full = pd.read_csv("/cluster/courses/cil/text-classification/data/train.csv")

# Data preprocessing TODO
We need to preprocess data. Step that come to my mind:
 - Remove word count outliers. (The vibe of a review comes across after 100 words or so).
 - We have german and english data. Should we translate everything to english
 - Should we search for smiley an insert text for them.
 - Should we search for ** (bold markers) and emphasize this word differenetly? 

# Build Validation Set
Todo: 

Currently we use 90% of the reviews for training, and the remaining 10% for validation
This should be optimized. Use k-fold validation or something like that to get most of our data as training and enable proper hyperparameter tuning later on.

In [20]:
train_df, val_df = train_test_split(
        train_full, test_size=0.1, stratify=train_full["label"], random_state=42
)

# Baselines
The current baseline implementation follows the structure:
$$\text{Output} = \text{Classifier}(\text{EmbeddingForTokens}(x))$$

In this specific case:
* **EmbeddingForTokens**: They use `CountVectorizer`, which creates a "Bag of Words" representation (counting word frequencies).
* **Classifier**: They use **Logistic Regression**.

We could implement additional baselines by choosing more complex **EmbeddingForTokens** methods (such as word-level embeddings or pre-trained vectors) and more sophisticated **Classifier** models (such as Random Forests or simple Neural Networks (MLP)).

# TRAIN PIPELINE

## Overview
A modular training pipeline that evaluates combinations of (vectorizer, classifier) pairs.
Defined across three files: embeddings.py, models.py, and train_loop.py.

## Step 1 — Vectorization (embeddings.py)
For classical vectorizers (BoW, TF-IDF), the signature is:
    (X_train, X_val) -> (X_train_embedded, X_val_embedded)
The vocabulary is fit on X_train only, then applied to both splits.

For pretrained embedding models (BERT, GloVe), sentences are encoded
independently with no fitting step:
    X -> X_embedded

## Step 2 — Classification (models.py)
Each classifier is a factory function that returns a fresh, untrained model instance.
Keeping them as functions (rather than instances) ensures each combination in the
loop starts from a clean state.

model is then used to fit X_train on Y_train: model.fit(X_train, Y_train)

## Step 3 — Train Loop (train_loop.py)
Takes a list of (vectorizer_fn, model_fn) tuples. For each combination it:
  1. Vectorizes the data
  2. Trains the classifier on the training embeddings X_train, Y_train
  3. Evaluates on both train and validation splits

Returns a list of result dicts, one per combination:
    {
        "vectorizer":          str,   # name of the vectorizer function
        "model":               str,   # name of the model factory
        "training_score":      float,
        "training_mae":        float,
        "training_accuracy":   float,
        "validation_score":    float,
        "validation_mae":      float,
        "validation_accuracy": float
    }

In [21]:
# # so that it reloads the modules every time we run this cell, so we always have the latest version of the code
# %load_ext autoreload
# %autoreload 2

# import importlib
# import train_loop
# importlib.reload(train_loop)
# from train_loop import train_loop

In [22]:
# NOTE: don't do GloVe, as only for English
# downlaod GloVe embeddings (100-dimensional vectors) and extract the file
# import urllib.request
# import zipfile

# url = "http://nlp.stanford.edu/data/glove.6B.zip"
# urllib.request.urlretrieve(url, "glove.6B.zip")
# with zipfile.ZipFile("glove.6B.zip", "r") as f:
#     f.extract("glove.6B.100d.txt")  # 100-dimensional vectors

## Simple baselines

As baselines, we first evaluate these combinations and then try to beat it later with more sophisticated approaches.
1) BoW  +  Logistic Regression
2) TF-IDF  +  Logistic Regression
3) multilingual pre-trained embeddings + Logistic Regression

Notes: MLP and Random Forest take way too long on such sparse and high dimensional embeddings as BoW and TF-IDF... Skipped for now

In [23]:
# import sys
# import subprocess
# subprocess.run([sys.executable, "-m", "pip", "install", "sentence-transformers"])

In [24]:
import os
os.chdir('/home/taekim/Garnella')

In [25]:
%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [26]:
import os
os.environ["PATH"] = "/cluster/courses/cil/envs/envs/text-classification/bin:" + os.environ.get("PATH", "")
import sys
print(sys.executable)

/cluster/courses/cil/envs/envs/text-classification/bin/python


In [27]:
import torch
print("PyTorch CUDA version:", torch.version.cuda)
print("GPU visible:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")

PyTorch CUDA version: 12.8
GPU visible: True
GPU name: NVIDIA GeForce RTX 5060 Ti


In [28]:
from embeddings import *
from models import *
from train_loop_caching import train_loop

# Cell 1: Embedding comparison (same model, different embeddings)
combinations_embeddings = [
    #(get_tfidf_embeddings, get_logistic_regression),
   # (get_multilingual_embeddings, get_logistic_regression),
    #(get_gemma_embeddings, get_logistic_regression),
    (get_qwen_embeddings, get_logistic_regression),
]
results_emb = train_loop(train_df, val_df, combinations_embeddings)
pd.DataFrame(results_emb).sort_values("validation_score", ascending=False)

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 28749.38it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 48.00 MiB. GPU 0 has a total capacity of 15.48 GiB of which 36.88 MiB is free. Including non-PyTorch memory, this process has 15.43 GiB memory in use. Of the allocated memory 15.17 GiB is allocated by PyTorch, and 55.90 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
# Cell 2: Model comparison (best embeddings, different classifiers)
combinations_models = [
    (get_qwen_embeddings, get_logistic_regression),
    (get_qwen_embeddings, get_linear_svm),
    (get_qwen_embeddings, get_mlp),
    (get_qwen_embeddings, get_xgboost),
    (get_qwen_embeddings, get_knn),
    (get_gemma_embeddings, get_mlp),
    (get_gemma_embeddings, get_xgboost),
]
results_models = train_loop(train_df, val_df, combinations_models)
pd.DataFrame(results_models).sort_values("validation_score", ascending=False)